# Zugriff auf HID Geräte aus Python unter Linux

z.B. die VC 870 Messgeräte an der Hochschule und meine Uni-T Messgeräte haben nur ein hid Interface. Leider keine echte USB Serial Schnittstelle.

Ich suchte eine Lösung die unter Linux und unter Windows gleichermaßen funktioniert.

## Python Paket und die Dokumentation

Das Paket heißt hidapi und es stellt hid und hidraw bereit. Beide mit gleicher Schnittstelle.

- Code https://github.com/trezor/cython-hidapi
- PyPi https://pypi.org/project/hidapi/

Benötigt wird die libusb.

## Windows

https://github.com/libusb/hidapi/releases

Herunterladen und im Python verzeichnis ablegen: C:\Users\mathias.moog\AppData\Local\Programs\Python\Python313

Anschließend das passende Paket installieren:

    pip install hidapi
  
Unter Windows steht nur das hid Interface zur Verfügung, nicht hidraw.

## Linux 

Regeln für die hidraw Geräte anpassen, siehe https://gitlab.com/CalcProgrammer1/OpenRGB/-/work_items/1985

    sudo nano /etc/udev/rules.d/51-usb-device.rules

Folgende Zeile hinzu

    SUBSYSTEMS=="hidraw", GROUP="plugdev", MODE="666"

Damm die Regeln neu laden

    sudo udevadm control --reload-rules

Jetzt wird noch die aktuelle hid implementierung benötigt. 

    sudo apt install python3-hid

Im Python Code

    import hidraw as hid


Damit funktioniert es!

## Fehlersuche ...


Zu Beginn hatte ich import hid wie unter Windows und wie im Beispiel zur Bibliothek verwendet. Das funktionierte weder mit udev rules noch mit root Rechten.

- https://manpages.ubuntu.com/manpages/questing/man3/Device::USB::FAQ.3pm.html
  Hm, da hat sich einiges getan. Aber auch diese Regeln helfen nicht.
- https://deepwiki.com/libusb/hidapi/6.3-udev-rules-for-linux 
  Unterscheidung hidraw und libusb Hilft auch nicht!

Als Root in Python ausprobiert. Gleicher Fehler.



Notizzettel Befehle

    groups
    grep ATT /etc/udev/rules.d/*
    sudo nano /etc/udev/rules.d/51-usb-device.rules 
    sudo udevadm control --reload-rules
    grep 1a86 /etc/udev/rules.d/*
    udevadm monitor
    udevadm info -a -n /dev/bus/usb/001/009

Meine getesteten udef Regeln

    # Ohne Regel lässt sich das Device nicht öffnen
    # Try for libusb used by python3-hidapi, device lässt sich öffnen, aber nicht lesen
    #SUBSYSTEM=="usb", ATTR{idVendor}=="1a86", ATTR{idProduct}=="e008", MODE="0666"
    #SUBSYSTEM=="usb", ATTR{idVendor}=="1a86", ATTR{idProduct}=="e008", MODE="0666",  GROUP="plugdev"
    #SUBSYSTEM=="usb", ATTRS{idVendor}=="1a86", ATTRS{idProduct}=="e008", TAG+="uaccess", GROUP="plugdev", MODE="666"
    #SUBSYSTEM=="usb", ATTRS{idVendor}=="1a86", ATTRS{idProduct}=="e008", MODE="0660", GROUP="plugdev", TAG+="uaccess", TAG+="udev-acl", SYMLINK+="ut803" 
    #SUBSYSTEM=="hidraw", ATTRS{idVendor}=="1a86", ATTRS{idProduct}=="e008", MODE="0660", GROUP="plugdev", TAG+="uaccess", TAG+="udev-acl", 
    # Does not work with libUSB, das hidraw device ist in der richtigen gruppe, aber ich komme nicht heran.
    #SUBSYSTEMS=="hidraw", GROUP="plugdev", MODE="0666"
    # Next trial, device lässt sich öffnen, aber nicht lesen
    #SUBSYSTEM=="usb", ENV{DEVTYPE}=="usb_device", MODE="0666"
    # Aus der Ubuntu Anleitung weder mode noch group funktionieren. Das Device lässt sich nicht einmal öffnen
    #SUBSYSTEM=="usb_device", MODE="0664", SYSFS{idVendor}=="1a86", SYSFS{idProduct}=="e008"
    # Aus der Ubuntu Anleitung, Egal ob mode oder Gruppe, es passt nicht.
    #SUBSYSTEM=="usb", MODE="0664", GROUP="plugdev"

Es ist wirklich nur dieses eine hidapi installiert

    pip list | grep hid
    hidapi                       0.15.0

Als root läuft hidraw:

    import hidraw
    dev = hidraw.device()
    dev.open(6790,57352)
    dev.get_product_string()

Mit der Rule 

    SUBSYSTEMS=="hidraw", GROUP="plugdev", MODE="0666"

Funktioniert es auch als user der in der plugdev gruppe ist.
    


### Recherche, wo kommen die Pakete her und was ist enthalten

- python-hidapi: https://launchpad.net/ubuntu/+source/python-hidapi python3-hidapi gibt es nicht !?
  In der Anleitung kommt heraus, das damit python3-hid gebaut wird !? Basierend auf cython3, repository git clone https://git.launchpad.net/ubuntu/+source/python-hidapi mit alten cython interface. In Setup.py https://github.com/libusb/hidapi, https://github.com/trezor/cython-hidapi

 - python3-hid: apt-cache show python3-hid  https://github.com/trezor/cython-hidapi, passt zum ersten Punkt. Dort Verweis auf https://pypi.org/project/hidapi/

 - python3-hidapi, apt-cache show python3-hidapi, keine Quelle, aber abhängigkeit python3-cffi, apt-cache show python3-cffi, ahttps://cffi.readthedocs.org/. Installiert, in python help(hidapi), keine Links zu quellen, nur verweis auf Code cat /usr/lib/python3/dist-packages/hidapi.py das basiert auf cffi, aber keine Referenz enthalten.





## hid 

Benötigt hidapi als externes Paket. Siehe pypi: https://pypi.org/project/hidapi/, Dokumentation https://trezor.github.io/cython-hidapi/api.html, Source Code https://github.com/trezor/cython-hidapi

    pip install hidapi

Unter Linux ist es komplett mit Abhängigkeiten als Paket enthalten

    sudo apt install python3-hid

    

In [1]:
import sys
if sys.platform.startswith('linux'):
    import hidraw as hid
else:
    import hid
import pandas as pd

## Geräte auflisten

In [11]:
H = hid.enumerate()
H

[{'path': b'\\\\?\\HID#VID_0B05&PID_19AF&MI_02#a&d29d7f0&0&0000#{4d1e55b2-f16f-11cf-88cb-001111000030}',
  'vendor_id': 2821,
  'product_id': 6575,
  'serial_number': '9876543210',
  'release_number': 256,
  'manufacturer_string': 'AsusTek Computer Inc.',
  'product_string': 'AURA LED Controller',
  'usage_page': 65394,
  'usage': 161,
  'interface_number': 2,
  'bus_type': 1},
 {'path': b'\\\\?\\HID#VID_413C&PID_2105#8&2462a981&0&0000#{4d1e55b2-f16f-11cf-88cb-001111000030}\\KBD',
  'vendor_id': 16700,
  'product_id': 8453,
  'serial_number': '',
  'release_number': 850,
  'manufacturer_string': 'Dell',
  'product_string': 'Dell USB Keyboard',
  'usage_page': 1,
  'usage': 6,
  'interface_number': 0,
  'bus_type': 1},
 {'path': b'\\\\?\\HID#VID_0461&PID_4D64#8&1235d71a&0&0000#{4d1e55b2-f16f-11cf-88cb-001111000030}',
  'vendor_id': 1121,
  'product_id': 19812,
  'serial_number': '',
  'release_number': 512,
  'manufacturer_string': '(Standardsystemgeräte)',
  'product_string': 'USB Opti

{'path': b'\\\\?\\HID#VID_1A86&PID_E008#9&2afa1e71&0&0000#{4d1e55b2-f16f-11cf-88cb-001111000030}',
  'vendor_id': 6790,
  'product_id': 57352,
  'serial_number': '',
  'release_number': 4864,
  'manufacturer_string': 'WCH.CN \x13',
  'product_string': 'USB to Serial',
  'usage_page': 65440,
  'usage': 1,
  'interface_number': 0,
  'bus_type': <BusType.USB: 1>}]

In [ ]:
Hdf = pd.json_normalize(H)
Hdf["path"] = Hdf["path"].apply(lambda x: x.decode("utf-8") if isinstance(x, bytes) else x)
Hdf.drop(columns=['release_number', 'interface_number', 'usage_page', 'usage', 'interface_number', 'bus_type'], inplace=True)
Hdf

,path,vendor_id,product_id,serial_number,manufacturer_string,product_string
0,\\?\HID#VID_0B05&PID_19AF&MI_02#a&d29d7f0&0&00...,2821,6575,9876543210,AsusTek Computer Inc.,AURA LED Controller
1,\\?\HID#VID_413C&PID_2105#8&2462a981&0&0000#{4...,16700,8453,,Dell,Dell USB Keyboard
2,\\?\HID#VID_0461&PID_4D64#8&1235d71a&0&0000#{4...,1121,19812,,(Standardsystemgeräte),USB Optical Mouse
3,\\?\HID#VID_1A86&PID_E008#9&2ca0d236&0&0000#{4...,6790,57352,,WCH.CN ,USB to Serial


In [26]:
Hdf.loc[3]["path"]

'\\\\?\\HID#VID_1A86&PID_E008#9&2ca0d236&0&0000#{4d1e55b2-f16f-11cf-88cb-001111000030}'

### Merker für verschiedene Geräte

- Optische Schnittstelle für VC870 Multimeter melden sich als vendor_id = 6790	product_id 57352		WCH.CN 	USB to Serial


# Abfragen

Wie im Beispielcode auf https://github.com/trezor/cython-hidapi unter try.py

In [13]:
# Device öffnen
dev = hid.device()
dev.open(6790,57352)

In [14]:
# Print some information about the device
print("Manufacturer: %s" % dev.get_manufacturer_string())
print("Product: %s" % dev.get_product_string())
print("Serial No: %s" % dev.get_serial_number_string())
print("Report descriptor: %s" % dev.get_report_descriptor())

Manufacturer: WCH.CN 
Product: USB to Serial
Serial No: 
Report descriptor: [6, 160, 255, 9, 1, 161, 1, 9, 1, 21, 0, 38, 255, 0, 117, 8, 149, 8, 129, 2, 9, 2, 21, 0, 38, 255, 0, 117, 8, 149, 8, 145, 2, 9, 3, 21, 0, 38, 255, 0, 117, 8, 149, 5, 177, 2, 192]


## VC 870

Mit optischer Schnittstelle, gleicher chip wie mein ut 803, siehe unten

Siehe Mein Ut803, HotexHe2225U.pdf

0 - report id, immer 0
1-4 - long baud rate, litte endian
5 - Immer den wert 3 senden

In [15]:
fr = bytes.fromhex('00 80 25 00 00 03')
print(fr)
dev.send_feature_report(fr)

b'\x00\x80%\x00\x00\x03'


6

Ein erster Test, es bit immer einzelne Frames in denen ggf. mal Daten stehen. Die beginnen immer mit 240

In [16]:
dev.read(20,3000)

[240, 0, 0, 0, 0, 0, 0, 0]

Lese bis Zeilenende und zeige an. Es sollte der Messwert in ascii Codierung erscheinen.

In [18]:
i=0
buffer = bytearray()
while i<300:
  b=dev.read(8)
  l=b[0]&0x07
  #buffer.extend(b[1:l+1])
  for j in range(1,l+1):
    buffer.append(b[j]&0x7F)
  #print(f"i={i}, l={l} b={b} buffer={buffer} len={len(buffer)}")
  if buffer.endswith(b'\r\n'):
    print("Ende erkannt")
    break
  i+=1

print(buffer.decode('ascii'))

Ende erkannt
000000080000000400500



In [ ]:
dev.close()

## UNI-T UT 803

Jetzt zu meinem UNI-T UT 803, das will nicht sofort reden. Es muss erst konfiguriert werden.

Konfigurieren, 19200 Baud = 0x4B00, Rest wie in HoitexHE2325U.pdf

In [7]:
fr = bytes.fromhex('00 00 4B 00 00 03')
print(fr)
dev.send_feature_report(fr)

b'\x00\x00K\x00\x00\x03'


6

In [8]:
dev.read(20,3000)

[240, 0, 0, 0, 0, 0, 0, 0]

Hm, es kommt immer ein leerer String. Der Timeout ist in ms.

In [9]:
b=[];
i=0;
while i<10 and len(b)<200:
  b.append(dev.read(20,300))
  i+=1

print(b)

[[240, 0, 0, 0, 0, 0, 0, 0], [240, 0, 0, 0, 0, 0, 0, 0], [240, 0, 0, 0, 0, 0, 0, 0], [240, 0, 0, 0, 0, 0, 0, 0], [240, 0, 0, 0, 0, 0, 0, 0], [240, 0, 0, 0, 0, 0, 0, 0], [240, 0, 0, 0, 0, 0, 0, 0], [240, 0, 0, 0, 0, 0, 0, 0], [240, 0, 0, 0, 0, 0, 0, 0], [240, 0, 0, 0, 0, 0, 0, 0]]


/*
 Aus UT 803.log
 *  Byte			6	5	4	3	2	1	0
 * [0]	Bereich		0	1	1	0	====siehe unten===
 * [1]	1. Ziffer	0	1	1	===========Ziffer=========
 * [2]	2. Ziffer	0	1	1	===========Ziffer=========
 * [3]	3. Ziffer	0	1	1	===========Ziffer=========
 * [4]	4. Ziffer	0	1	1	===========Ziffer=========
 * [5]	Schalterst.	0	1	1	========siehe unten=======
 * [6]	Info		0	1	1	°C	NEG	LoBat	OVL	// °C-Bit ist immer gesetzt außer bei °F
 * [7]	Info2		0	1	1	HOLD	MAX	MIN	0	// Rel vermutlich geplant, aber keine Taste vorhanden
 * [8]	Kopplung	0	1	1	DC	AC	AUTO	0	// DC und AC zwar kombiniert am Gerät, jedoch keine Übertragung!
 * [9]	'\r'		0	0	0	1	1	0	1
 * [10]	'\n'		0	0	0	1	0	1	0

 * LONIBBLE(Byte[5]):	B	3	6	2	D	F	9	4	4	E       5   
 * Bereich (Byte[0]):	V	Ohm	F	Hz	µA	mA	A	°C	°F	hFE    Diode
 *          '0'		6.000	600.0	6.000n	6000	600.0µ	60.00m	10.00	1000	1800	6000
 *          '1'		60.00	6.000k	60.00n	60.00k	6000µ	600.0m	-	-	-	-
 *          '2'		600.0	60.00k	600.0n	600.0k	-	-	-	-	-	-
 *          '3'		1000	600.0k	6.000µ	6.000M	-	-	-	-	-	-
 *          '4'		600.0m	6.000M	60.00µ	60.00M	-	-	-	-	-	-
 *          '5'		-	60.00M	600.0µ	-	-	-	-	-	-	-
 *          '6'		-	-	6.000m	-	-	-	-	-	-	-
 *          '7'		-	-	-	-	-	-	-	-	-	-
 * 
 * Schalterstellung (Byte[5]) und etwaige Abtastrate:
 * 	'1'	Diode	10 Sa/s
 * 	'2'	Hz	2 Sa/s
 * 	'3'	Ohm	2 Sa/s
 * 	'4'	°F/°C	2 Sa/s
 * 	'5'	Pieps	10 Sa/s
 * 	'6'	F	im Bereich 6 mF sehr niedrige Rate, etwa 0,2 Sa/s {5 s/Sa}
 * 	'9'	A	3 Sa/s
 * 	';'(3B)	V	2 Sa/s
 * 	'='(3D)	µA	3 Sa/s
 * 	'>'(3E)	hFE	2 Sa/s
 * 	'?'(3F)	mA	3 Sa/s
 */

 Es gibt einen kleinen Puffer. Doppelt lesen sonst ggf. alter Wert.

 Umsetzung in Java Elektro/Programme/DMM/DMM/src/dmm/UT803decode.java

In [10]:
LAENGE=11
i=0
buffer = bytearray()
while i<300:
  b=dev.read(8)
  l=b[0]&0x07
  #buffer.extend(b[1:l+1])
  for j in range(1,l+1):
    buffer.append(b[j]&0x7F)
  #print(f"i={i}, l={l} b={b} buffer={buffer} len={len(buffer)}")
  if buffer.endswith(b'\r\n'):
    print("Ende erkannt")
    break
  i+=1

if len(buffer)==2*LAENGE:
    print(buffer.decode('ascii'))
    a=buffer[0:LAENGE-2]
    b=buffer[LAENGE:2*LAENGE-2]
    if a==b:
        print("OK") 
    else:
        print("Fehler, unterschiedliche Werte empfangen")

# Einheiten, auf Index aufpassen!
utEinheit = [ "", "", "Hz", "Ohm", "°C", "°C", "F", "", "", "A", "", "V", "", "A", "hFE", "A"]
# Umrechenfaktoren
NA=float('nan')
utFaktor = [
    [NA, NA, 1e0, 0.1e0, 1e0, 0.1e0, 0.001e-9, NA, NA, 0.01e0, NA, 0.001e0, NA, 0.1e-6, 1, 0.01e-3],
    [NA, NA, 0.01e3, 1e0, NA, 1e0, 0.01e-9, NA, NA, NA, NA, 0.01e0, NA, 1e-6, NA, 0.1e-3],
    [NA, NA, 0.1e3, 0.01e3, NA, 0.01e3, 0.1e-9, NA, NA, NA, NA, 0.1e0, NA, NA, NA, NA],
    [NA, NA, 1e3, 0.1e3, NA, 0.1e3, 0.001e-6, NA, NA, NA, NA, 1e0, NA, NA, NA, NA],
    [NA, NA, 0.01e6, 1e3, NA, 1e3, 0.01e-6, NA, NA, NA, NA, 0.1e-3, NA, NA, NA, NA],
    [NA, NA, NA, 0.01e6, NA, 0.01e6, 0.1e-6, NA, NA, NA, NA, NA, NA, NA, NA, NA],
    [NA, NA, NA, NA, NA, NA, 0.1e-6, NA, NA, NA, NA, NA, NA, NA, NA, NA],
    [NA, NA, NA, NA, NA, NA, NA, NA, NA, NA, NA, NA, NA, NA, NA, NA]]

# split line
bereich = a[0] & 0x7
wert = a[1:5].decode('ascii')
schalter = a[5] & 0xf
info = a[6] & 0xf
kopplung = a[8] & 0xf
print(f"bereich={bereich} wert={wert} schalter={schalter} info={info} kopplung={kopplung}")

# extract unit
unit = utEinheit[schalter];


# Numerischen Wert zur entsprechenden Einheit
value = float(wert)*utFaktor[bereich][schalter]
# Vorzeichen aus info herausfummeln
if info&4:
        value = -value;
# Kopplung mit an die Einheit anhängen
if info&1:
        value = NA;
if kopplung&4:
        unit = [ unit, " (AC)" ];
if kopplung&8:
        unit = [ unit, " (DC)" ];
print(f"Wert={value} {unit}")


Ende erkannt
00000;<0:
00000;<0:

OK
bereich=0 wert=0000 schalter=11 info=12 kopplung=10
Wert=-0.0 ['V', ' (DC)']


In [11]:
b'\r\n'

b'\r\n'

In [12]:
int.from_bytes(b'\r')

13

In [13]:
#dev.get_input_report(2,11)
# Funktioniert nicht.

In [19]:
dev.close()